# LuantiVoxel — automated GPU visual validation

Opens the real game, renders it from fixed viewpoints at **1920x1080**, captures the
actual framebuffer, and packages everything into a ZIP you can download.

No playing, no clicking, no remote desktop. Press **Runtime > Run all**.

**It refuses to lie.** If the GPU cannot be positively identified as hardware —
llvmpipe, SwiftShader, swrast, anything software — the run stops with
`SOFTWARE_RENDERING_DETECTED` and produces no screenshots. A software render of a
voxel landscape looks perfectly plausible and is worthless as evidence of what the
game looks like on a GPU, so it is treated as a failure rather than a result.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG — change REPO_URL if you have a fork
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/Daniel-debug-boop/luanti.git"
REF      = "master"
GODOT_VERSION = "4.4-stable"

RESOLUTION  = "1920x1080"
FRAMES      = 120
WARMUP      = 30
OUTPUT_DIR  = "render-test-results"

# GPU detection is a hard gate. Leave this True.
REQUIRE_HARDWARE_GPU = True

import os, sys, subprocess, json, textwrap

def sh(cmd, check=True, **kw):
    print(f"$ {cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, text=True,
                       capture_output=True, **kw)
    if p.stdout: print(p.stdout)
    if p.stderr: print(p.stderr, file=sys.stderr)
    if check and p.returncode != 0:
        raise SystemExit(f"command failed ({p.returncode}): {cmd}")
    return p

print("config ok")

## 1 — Environment

Prints what this machine actually is. It does **not** assume NVIDIA: Colab may hand you
a T4, an L4, an A100, or a CPU-only runtime, and the pipeline has to work with whichever
device it is given or refuse.

In [ ]:
print("=" * 70)
print("OS        :", sh("uname -a", check=False).stdout.strip())
print("python    :", sys.version.split()[0])

# Every probe is optional and guarded. Absence of one is information, not an error.
for tool, args in [("nvidia-smi", "nvidia-smi"),
                  ("lspci", "lspci | grep -iE 'vga|3d|display' || true"),
                  ("dri", "ls -l /dev/dri 2>&1 || true"),
                  ("glxinfo", "which glxinfo || echo 'not installed'")]:
    print(f"\n--- {tool} ---")
    print(sh(args, check=False).stdout.strip() or "(nothing)")

## 2 — Clone the repository

In [ ]:
REPO_DIR = "/content/luantivoxel"
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --depth 1 --branch {REF} {REPO_URL} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull --ff-only origin {REF}")

GIT_COMMIT = sh(f"git -C {REPO_DIR} rev-parse HEAD").stdout.strip()
print("commit:", GIT_COMMIT)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 3 — Dependencies

The game is the **Godot 4.4 client** in `godot_client/`, so the C++ engine in `src/` is
not needed and is not compiled — that would be minutes of build time for a component the
benchmark does not run.

What is needed is the Godot binary, its **export templates** (to produce a standalone
build rather than a project folder), and Xvfb plus Mesa for a display to render into.
All of it comes from the project's own tooling — see `godot_client/tools/build_release.sh`.

In [ ]:
sh("apt-get update -qq && apt-get install -y -qq "
   "xvfb libgl1-mesa-dri libglx-mesa0 libxcursor1 libxinerama1 "
   "libxrandr2 libxi6 libxext6 libx11-6 unzip wget", check=False)

GODOT_DIR = "/opt/godot"
os.makedirs(GODOT_DIR, exist_ok=True)
GODOT = f"{GODOT_DIR}/godot"

if not os.path.exists(GODOT):
    url = (f"https://github.com/godotengine/godot/releases/download/"
           f"{GODOT_VERSION}/Godot_v{GODOT_VERSION}_linux.x86_64.zip")
    sh(f"wget -q {url} -O /tmp/godot.zip && unzip -o -q /tmp/godot.zip -d {GODOT_DIR}")
    sh(f"mv {GODOT_DIR}/Godot_v{GODOT_VERSION}_linux.x86_64 {GODOT} && chmod +x {GODOT}")

print(sh(f"{GODOT} --version").stdout.strip())

## 4 — Build

Uses the project's own `build_release.sh`, which exports the Linux build **and
smoke-tests the resulting binary**, failing if it does not reach a ready state or logs a
script error. A build that starts and then falls over never reaches the render test.

In [ ]:
TPL_DIR = os.path.expanduser(
    "~/.local/share/godot/export_templates/4.4.stable")
os.makedirs(TPL_DIR, exist_ok=True)

if not os.path.exists(f"{TPL_DIR}/linux_release.x86_64"):
    tpl = (f"https://github.com/godotengine/godot/releases/download/"
           f"{GODOT_VERSION}/Godot_v{GODOT_VERSION}_export_templates.tpz")
    sh(f"wget -q {tpl} -O /tmp/tpl.tpz && unzip -o -q /tmp/tpl.tpz -d /tmp/tpl")
    for f in ("linux_release.x86_64", "linux_debug.x86_64", "version.txt"):
        sh(f"cp /tmp/tpl/templates/{f} {TPL_DIR}/")

sh(f"sh godot_client/tools/build_release.sh {GODOT}")

BUILD_DIR = os.path.join(REPO_DIR, "build", "luantivoxel")
EXE = os.path.join(BUILD_DIR, "luantivoxel.x86_64")
PCK = os.path.join(BUILD_DIR, "luantivoxel.pck")
print("exe:", EXE, os.path.exists(EXE))
print("pck:", PCK, os.path.exists(PCK))

## 5 — GPU verification (hard gate)

Starts a virtual display and asks the **actual game binary** what adapter it is on.
This is the same code path the render test uses, not a separate check that could
disagree with it.

In [ ]:
os.environ["DISPLAY"] = ":99"
os.environ.setdefault("HOME", "/root")
sh("pkill Xvfb 2>/dev/null; Xvfb :99 -screen 0 1920x1080x24 >/tmp/xvfb.log 2>&1 &")
import time; time.sleep(3)

def gpu_gate():
    """Ask the game itself, trying each driver in turn."""
    best = ("", "")
    for drv in DRIVERS:
        print(f"\n########## trying rendering driver: {drv} ##########")
        out = os.path.join(os.getcwd(), "_gpuprobe")
        sh(f"rm -rf {out}")
        try:
            p = subprocess.run(
                [EXE, "--rendering-driver", drv, "--resolution", RESOLUTION,
                 "--", "--render-test", "--frames", "1",
                 "--warmup-frames", "0", "--output", out],
                text=True, capture_output=True, timeout=900,
                env={**os.environ})
        except subprocess.TimeoutExpired:
            print(f"{drv}: timed out")
            continue
        txt = p.stdout + p.stderr
        for line in txt.splitlines():
            if "render-test" in line:
                print(line)
        report = os.path.join(out, "gpu-info.txt")
        body = open(report).read() if os.path.exists(report) else ""
        cls = ""
        for line in body.splitlines():
            if line.startswith("classification"):
                cls = line.split(None, 1)[1].strip()
        if body:
            print(body)
        if cls == "hardware":
            print(f"\n{drv} gave real hardware acceleration. Using it.\n")
            return drv, cls
        best = (drv, cls)
    return best

DRIVERS = ["vulkan", "opengl3"]
chosen, cls = gpu_gate()
os.environ["RENDER_DRIVER"] = chosen or "vulkan"
print(f"\ndriver chosen = {os.environ['RENDER_DRIVER']!r}, "
      f"classification = {cls!r}")

if REQUIRE_HARDWARE_GPU and cls != "hardware":
    raise SystemExit(
        "Hardware GPU rendering was not detected. Refusing to produce a "
        f"misleading final visual-validation result (classification={cls!r}).\n"
        "\nFix: Runtime > Change runtime type > GPU, then restart. 'GPU' in the "
        "Colab picker does not always mean a device is attached, and Colab GPU "
        "availability varies. Cell 1 shows what the runtime actually has.")

## 6 — Render test

1920x1080, four fixed viewpoints, real framebuffer, real assets, real shaders.

In [ ]:
sh(f"rm -rf {OUTPUT_DIR}")
cmd = [EXE, "--rendering-driver", os.environ.get("RENDER_DRIVER", "vulkan"),
       "--resolution", RESOLUTION, "--", "--render-test",
       "--scene", "benchmark", "--resolution", RESOLUTION,
       "--frames", str(FRAMES), "--warmup-frames", str(WARMUP),
       "--all-cameras", "--no-ui", "--output", OUTPUT_DIR]
print("$ " + " ".join(cmd))
proc = subprocess.run(cmd, text=True, capture_output=True, timeout=3600,
                      env={**os.environ})
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise SystemExit(f"render test failed with exit code {proc.returncode}")
print("render test exited 0")

## 7 — Validate

Refuses to package a result that is incomplete or the wrong size. A screenshot that is
not genuinely 1920x1080 is worse than no screenshot, because it looks like one.

In [ ]:
from PIL import Image

W, H = (int(x) for x in RESOLUTION.split("x"))
cap_dir = os.path.join(OUTPUT_DIR, "captures")
shots = sorted(f for f in os.listdir(cap_dir) if f.endswith(".png")) \
    if os.path.isdir(cap_dir) else []
problems = []

if not shots:
    problems.append("SCREENSHOT_CAPTURE_FAILURE: no captures produced")
for s in shots:
    p = os.path.join(cap_dir, s)
    w, h = Image.open(p).size
    ok = (w, h) == (W, H)
    print(f"  {'ok  ' if ok else 'BAD '} {s:<22} {w}x{h}")
    if not ok:
        problems.append(f"INVALID_RESOLUTION: {s} is {w}x{h}, expected {W}x{H}")

for required in ("metadata.json", "performance.json", "performance.txt",
                 "gpu-info.txt", "render-log.txt"):
    p = os.path.join(OUTPUT_DIR, required)
    if not os.path.exists(p):
        problems.append(f"missing artefact: {required}")
    else:
        print(f"  ok   {required}")

meta = json.load(open(os.path.join(OUTPUT_DIR, "metadata.json")))
if not meta.get("hardware_acceleration"):
    problems.append("HARDWARE_GPU: FAIL — metadata does not claim acceleration")

if problems:
    raise SystemExit("VALIDATION FAILED:\n  " + "\n  ".join(problems))
print("\nvalidation passed")

## 8 — Package

In [ ]:
import datetime, zipfile

sh(f"echo {GIT_COMMIT} > {OUTPUT_DIR}/git-commit.txt")
sh(f"({GODOT} --version; echo; nvidia-smi 2>/dev/null || true) "
   f"> {OUTPUT_DIR}/build-info.txt")

README = '''LuantiVoxel GPU render test
============================
Automated visual validation of the Godot 4.4 client.

These are real frames from the shipping renderer at 1920x1080 on a hardware
GPU. An llvmpipe/software result is NOT a valid visual validation and this
pipeline refuses to produce one.

The seed is recorded in metadata.json; the same seed reproduces the same
terrain on any machine, so two runs are directly comparable.
'''
open(os.path.join(OUTPUT_DIR, "README.txt"), "w").write(README)

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
ZIP = f"LUANTIVOXEL_GPU_RENDER_TEST_{stamp}.zip"

# Only validation artefacts. No build dirs, no .git, no caches.
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _dirs, files in os.walk(OUTPUT_DIR):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, os.path.dirname(OUTPUT_DIR)))

print("zip:", ZIP, os.path.getsize(ZIP), "bytes")
for n in sorted(zipfile.ZipFile(ZIP).namelist()):
    print("  ", n)

## 9 — Results

In [ ]:
perf = json.load(open(os.path.join(OUTPUT_DIR, "performance.json")))
fps = perf.get("average_fps")

print("=" * 56)
print("LUANTIVOXEL GPU RENDER TEST")
print("=" * 56)
print(f"GPU                : {meta.get('gpu_vendor')} / {meta.get('gpu_name')}")
print(f"Graphics API       : {meta.get('graphics_api')}")
print(f"Rendering device   : {meta.get('rendering_device')}")
print(f"Hardware accel     : {'PASS' if meta.get('hardware_acceleration') else 'FAIL'}")
print(f"Resolution         : {meta.get('resolution')}")
print(f"Scene / seed       : {meta.get('scene')} / {meta.get('seed')}")
print(f"Frames             : {perf.get('frames_measured')} measured, "
      f"{perf.get('warmup_frames')} warm-up")
print(f"Average frame time : "
      f"{perf['average_frame_ms']:.2f} ms" if perf.get('average_frame_ms')
      else "Average frame time : unavailable")
print(f"Average FPS        : {fps:.2f}" if fps else "Average FPS        : unavailable")
print(f"GPU frame time     : {perf.get('gpu_frame_ms') or 'unavailable'}")
print(f"Screenshots        : {', '.join(shots)}")
print(f"Commit             : {GIT_COMMIT[:12]}")
print("=" * 56)
from IPython.display import display, Image as IPImage, HTML
display(HTML(f'<a href="{ZIP}" download>Download {ZIP}</a>'))
for s in shots:
    display(IPImage(filename=os.path.join(cap_dir, s)))